# BBO Capstone: Reproducible Surrogate Optimisation

This notebook is a reviewer-friendly walkthrough of the final retrospective analysis. It uses the documented observation history, fits one transparent Gaussian-process surrogate per function, and compares acquisition-driven next-query proposals. It does not claim to recreate every historical submission.

## Method

1. Load the 88 documented query-output pairs.
2. Rank-normalise outputs within each function.
3. Fit a small fixed-kernel Gaussian-process surrogate.
4. Generate global and local candidate points.
5. Use UCB, EI, or posterior variance to select an unseen point.
6. Report leave-one-out rank error as a small-data stability check.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from bbo_capstone.data import format_query, load_datasets
from bbo_capstone.experiment import run_experiment

In [ ]:
datasets = load_datasets()
[(dataset.name, dataset.dimension, len(dataset.y)) for dataset in datasets]

In [ ]:
results = [run_experiment(dataset, acquisition='ucb') for dataset in datasets]

for result in results:
    print(f'{result.function} ({result.dimension}D)')
    print('  best observed output:', f'{result.best_observed_value:.8g}')
    print('  UCB proposal:        ', format_query(result.proposal))
    print('  predicted rank:      ', f'{result.predicted_rank:.3f}')
    print('  uncertainty:         ', f'{result.predicted_uncertainty:.3f}')
    print('  LOO rank MAE:        ', f'{result.loo_rank_mae:.3f}')
    print()

In [ ]:
comparison = {}
for acquisition in ('ucb', 'ei', 'variance'):
    comparison[acquisition] = [
        (dataset.name, format_query(run_experiment(dataset, acquisition=acquisition).proposal))
        for dataset in datasets
    ]
comparison

## Interpretation

UCB is the default because it preserves a controlled exploration term while still rewarding high predicted ranks. EI is more improvement-focused, while posterior variance intentionally targets uncertainty. With only 11 observations per function, differences between these recommendations should be read as useful experimental hypotheses, not as proof of a global solution.